# Prototipo — XGBoost para predecir NDVI/EVI

Segundo modelo candidato (después de Random Forest), para cerrar el pendiente de "comparar alternativas de modelos" del proyecto. Este notebook es exploratorio a propósito — corre aquí primero, con visibilidad completa de cada paso, antes de formalizarlo en `models/xgboost_model.py` siguiendo la misma estructura que `models/random_forest.py`.

Reutiliza `models/_experiment_utils.py` (mismo split cronológico, mismo `prepare_features`, mismas métricas) para que el resultado sea directamente comparable con Random Forest — no reinventa esa lógica.

**Este notebook NO escribe en `results/`** (ni `experiment_log.csv` ni `model_comparison.csv`) — eso queda para cuando se formalice en `models/xgboost_model.py`, igual que se hizo con Random Forest. Aquí solo se explora y se decide si vale la pena formalizarlo.

Usa `xgboost` (ya instalado, v3.1.1) — `lightgbm` no está instalado en esta máquina, así que se dejó fuera por ahora.

Cambia `Y_VARIABLE` en la celda de parámetros para explorar `ndvi` o `evi`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT / "models"))

from _experiment_utils import prepare_features, split_train_val_test, compute_metrics, REGIONS

%matplotlib inline

In [ ]:
# Parametros del notebook
Y_VARIABLE = "ndvi"  # cambiar a "evi" para explorar la otra variable

# Grid moderado a proposito -- son mas hiperparametros que Random Forest
# (learning_rate ademas de n_estimators/max_depth), y con pocas filas de
# entrenamiento (1030) no hace falta un grid enorme para ver si el modelo
# tiene potencial.
PARAM_GRID = {
    "n_estimators": [100, 200, 400],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
}
RANDOM_STATE = 42
N_CV_SPLITS = 5

## 1. Cargar dataset y split cronológico (mismo que Random Forest)

In [ ]:
DATASET_PATH = REPO_ROOT / "data" / "processed" / "dataset_modelo.csv"
df = pd.read_csv(DATASET_PATH, parse_dates=["window_start", "window_end"])

train_val, test_final = split_train_val_test(df, fecha_col="window_start", frac_test_final=0.15)
fecha_corte = test_final["window_start"].min()

## 2. Preparar features (X, y) para `Y_VARIABLE`

Mismas exclusiones que Random Forest: el otro índice de vegetación, `region` cruda (reemplazada por dummies), `et_resolution`, columnas de fecha/ventana.

In [ ]:
X_train, y_train = prepare_features(train_val, Y_VARIABLE)
X_test, y_test = prepare_features(test_final, Y_VARIABLE)
region_test = test_final["region"].reset_index(drop=True)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
X_train.head()

## 3. Baseline (hiperparámetros por defecto)

`XGBRegressor` acepta `NaN` nativamente en el split de árboles (verificado aparte) — igual que `RandomForestRegressor`, no hace falta imputar los ~0.17% de nulos reales en `et_mm_lag1`.

In [ ]:
baseline = XGBRegressor(random_state=RANDOM_STATE, n_jobs=1)
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)
r2_b, rmse_b, pearson_b = compute_metrics(y_test, y_pred_baseline)
print(f"baseline -> R2={r2_b:.4f}  RMSE={rmse_b:.4f}  Pearson={pearson_b:.4f}")

## 4. GridSearchCV con `TimeSeriesSplit` (mismo esquema que Random Forest)

Igual que en `random_forest.py`: el grid corre solo sobre `train_val`, con folds crecientes y cronológicos — `test_final` no se toca hasta evaluar el modelo ya tuneado.

In [ ]:
grid = GridSearchCV(
    estimator=XGBRegressor(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=PARAM_GRID,
    cv=TimeSeriesSplit(n_splits=N_CV_SPLITS),
    scoring="r2",
    n_jobs=-1,
)
grid.fit(X_train, y_train)
cv_results = pd.DataFrame(grid.cv_results_)

tuned = grid.best_estimator_
y_pred = tuned.predict(X_test)
r2_t, rmse_t, pearson_t = compute_metrics(y_test, y_pred)

print("best_params:", grid.best_params_)
print(f"mejor mean_test_score en CV (train_val): {cv_results['mean_test_score'].max():.4f}")
print(f"tuned en test_final                    -> R2={r2_t:.4f}  RMSE={rmse_t:.4f}  Pearson={pearson_t:.4f}")

## 5. Importancia de features (top 10)

In [ ]:
importancias = pd.Series(tuned.feature_importances_, index=X_train.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
importancias.head(10).sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"XGBoost — top 10 features más importantes ({Y_VARIABLE})")
plt.tight_layout()
plt.show()

importancias.head(10)

## 6. Real vs. predicho en `test_final`

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for region in REGIONS:
    mask = (region_test == region).values
    ax.scatter(y_test.values[mask], y_pred[mask], alpha=0.6, label=region)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, color="black", linestyle="--", linewidth=1, label="y = x (predicción perfecta)")
ax.set_xlabel(f"{Y_VARIABLE} real")
ax.set_ylabel(f"{Y_VARIABLE} predicho")
ax.set_title("XGBoost — real vs. predicho (test_final, tuned)")
ax.legend()
plt.show()

## 7. Métricas del modelo tuned por región

In [ ]:
for region in REGIONS:
    mask = (region_test == region).values
    r2_r, rmse_r, pearson_r = compute_metrics(y_test[mask], y_pred[mask])
    print(f"  {region} (n={mask.sum()}): R2={r2_r:.4f}  RMSE={rmse_r:.4f}  Pearson={pearson_r:.4f}")

## 8. Comparación contra Random Forest

Lee el resultado tuned de Random Forest ya guardado en `results/model_comparison.csv` (corrida con el dataset ya corregido por el filtro `SummaryQA`) y lo pone al lado del XGBoost tuned de este notebook, para `Y_VARIABLE`.

In [ ]:
comparison_path = REPO_ROOT / "results" / "model_comparison.csv"
model_comparison = pd.read_csv(comparison_path)
rf_row = model_comparison[
    (model_comparison["modelo"] == "random_forest") & (model_comparison["y_variable"] == Y_VARIABLE)
]

print(f"Comparación tuned, y_variable={Y_VARIABLE}:")
print(f"{'':15s} {'R2':>10s} {'RMSE':>10s} {'Pearson':>10s}")
if not rf_row.empty:
    row = rf_row.iloc[0]
    print(f"{'random_forest':15s} {row['r2']:10.4f} {row['rmse']:10.4f} {row['pearson']:10.4f}")
else:
    print("  (no hay fila de random_forest para esta variable en results/model_comparison.csv)")
print(f"{'xgboost':15s} {r2_t:10.4f} {rmse_t:10.4f} {pearson_t:10.4f}")

## Notas / próximos pasos

- Corre este notebook para `Y_VARIABLE = "ndvi"` y luego para `"evi"` (cambiando la celda de parámetros) antes de decidir si XGBoost vale la pena formalizar.
- Si XGBoost mejora a Random Forest en ambas variables (o al menos no empeora y aporta algo — p. ej. mejor R² o features más interpretables), el siguiente paso es portar esta lógica a `models/xgboost_model.py` con la misma estructura que `models/random_forest.py`: loop sobre `["ndvi", "evi"]` en una sola ejecución, `log_run` para baseline y tuned, `upsert_model_comparison`, y guardar `cv_results_` en `results/grid_search_raw/xgboost_{y_variable}.csv`.
- Si se decide formalizar, considerar ampliar el grid de hiperparámetros (`subsample`, `colsample_bytree`, `min_child_weight`) una vez que se sepa que la familia de modelo vale la pena — no antes, para no gastar tiempo de cómputo explorando hiperparámetros de un modelo que quizás no se adopte.